# 🧱 CrackSense – CNN Crack Classifier
**Dataset:** [CrackVision12K](https://www.kaggle.com/datasets/vangiap/crackvision12k)  
**Goal:** Binary classification (crack / no_crack) + intensity grading (HIGH / MEDIUM / LOW)  
**Model:** Custom 4-block CNN  

---

## 1. Setup & Imports

In [ ]:
import os, sys, warnings, math, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

# ===== PROJECT ROOT DETECTION =====
# This handles: local execution, cloud environments (Colab, Kaggle), and different CWDs
# 
# MANUAL OVERRIDE: Uncomment and set this if auto-detection fails
# PROJECT_ROOT = Path('/path/to/your/project')

cwd = Path.cwd().resolve()
PROJECT_ROOT = None

# Strategy 1: Search in current directory and parents
if PROJECT_ROOT is None:
    PROJECT_ROOT = next(
        (p for p in [cwd, *cwd.parents] if (p / 'src' / 'utils' / 'config.py').exists()),
        None,
    )

# Strategy 2: Search for src/utils/config.py recursively from common cloud paths
if PROJECT_ROOT is None:
    search_paths = [Path('/content'), Path('/kaggle/input'), Path('/kaggle/working'), Path.home() / 'work']
    for search_path in search_paths:
        if search_path.exists():
            try:
                found = next(search_path.rglob('src/utils/config.py'), None)
                if found:
                    PROJECT_ROOT = found.parent.parent.parent
                    break
            except (PermissionError, OSError):
                continue

# Strategy 3: Check if we can infer from uploaded data in Colab
if PROJECT_ROOT is None and Path('/content/drive/MyDrive').exists():
    try:
        found = next(Path('/content/drive/MyDrive').rglob('src/utils/config.py'), None)
        if found:
            PROJECT_ROOT = found.parent.parent.parent
    except (PermissionError, OSError):
        pass

# If still not found, provide helpful error
if PROJECT_ROOT is None:
    print('\n❌ PROJECT ROOT NOT FOUND')
    print(f'Current working directory: {cwd}')
    print(f'Searching for: src/utils/config.py')
    print('\nTo fix this:')
    print('1. Option A (local): Run from project root or subdirectory')
    print('2. Option B (Colab): Upload project to Drive and mount, OR use: !git clone <repo>')
    print('3. Option C (Kaggle): Upload project data or use Input datasets')
    print('4. Option D (manual): Uncomment and edit the PROJECT_ROOT line above')
    print('\nSearched in:')
    print(f'  - {cwd} and parents')
    print(f'  - /content (Kaggle/Colab)')
    print(f'  - /kaggle/working (Kaggle)')
    print(f'  - {Path.home()} (home)')
    raise RuntimeError(
        f'Could not locate project with src/utils/config.py from {cwd}'
    )

# Add to sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'✅ Project root: {PROJECT_ROOT}')

from src.utils.config import (
    KAGGLE_DATASET_HANDLE, CNN_IMG_SIZE, CNN_BATCH_SIZE,
    CNN_EPOCHS, LEARNING_RATE, VALIDATION_SPLIT,
    INTENSITY_HIGH, INTENSITY_MEDIUM, CNN_MODEL_PATH, SAVED_MODELS_DIR
)
from src.models.cnn_model import build_cnn_classifier

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')


## 2. Download Dataset via Kaggle Hub

In [ ]:
import kagglehub

# Auto-downloads dataset and caches locally.
# On first run, set KAGGLE_USERNAME and KAGGLE_KEY environment variables,
# or place ~/.kaggle/kaggle.json
print(f'Downloading dataset: {KAGGLE_DATASET_HANDLE} ...')
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_HANDLE)
dataset_path = Path(dataset_path)
print(f'Dataset cached at: {dataset_path}')

## 3. Explore Dataset Structure

In [ ]:
# Inspect directory layout
for root, dirs, files in os.walk(dataset_path):
    depth = root.replace(str(dataset_path), '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 2:
        subindent = '  ' * (depth + 1)
        for f in files[:5]:
            print(f'{subindent}{f}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more')

In [ ]:
# Locate positive (crack) and negative (no_crack) directories
# CrackVision12K typically has: Positive/ and Negative/ (or Crack/ and Non-Crack/)
# We auto-detect the class folders:
all_subdirs = [d for d in dataset_path.rglob('*') if d.is_dir()]

# Find the two-class root
data_root = None
for d in all_subdirs:
    children = [c for c in d.iterdir() if c.is_dir()]
    if len(children) == 2:
        data_root = d
        class_dirs = {c.name.lower(): c for c in children}
        break

assert data_root is not None, 'Could not find a two-class folder. Check dataset structure.'
print(f'Data root: {data_root}')
print(f'Classes found: {list(class_dirs.keys())}')

In [ ]:
# Count images per class
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def count_images(folder):
    return sum(1 for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXTS)

class_counts = {name: count_images(path) for name, path in class_dirs.items()}
print('Image counts per class:')
for k, v in class_counts.items():
    print(f'  {k}: {v:,}')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(class_counts.keys(), class_counts.values(),
               color=['#ef4444', '#22c55e'])
ax.bar_label(bars, padding=3)
ax.set_title('Class Distribution – CrackVision12K', fontweight='bold')
ax.set_ylabel('Image count')
plt.tight_layout()
plt.show()

In [ ]:
# Sample images
import random
from PIL import Image as PILImage

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, (cls_name, cls_path) in enumerate(class_dirs.items()):
    imgs = [f for f in Path(cls_path).rglob('*') if f.suffix.lower() in IMG_EXTS]
    sample = random.sample(imgs, min(6, len(imgs)))
    for col, img_path in enumerate(sample):
        im = PILImage.open(img_path).convert('RGB')
        axes[row, col].imshow(im)
        axes[row, col].set_title(cls_name, fontsize=9)
        axes[row, col].axis('off')
plt.suptitle('Sample Images from CrackVision12K', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Data Pipeline (ImageDataGenerator)

In [ ]:
IMG_H, IMG_W = CNN_IMG_SIZE

# Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    shear_range=0.1,
    fill_mode='reflect',
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
)

train_gen = train_datagen.flow_from_directory(
    str(data_root),
    target_size=(IMG_H, IMG_W),
    batch_size=CNN_BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42,
)

val_gen = val_datagen.flow_from_directory(
    str(data_root),
    target_size=(IMG_H, IMG_W),
    batch_size=CNN_BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=42,
)

class_map = {v: k for k, v in train_gen.class_indices.items()}
print('Class index:', train_gen.class_indices)
print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)}')

## 5. Build & Compile CNN Model

In [ ]:
model = build_cnn_classifier()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc'),
    ]
)

model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

## 6. Train CNN

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_auc', patience=8, restore_best_weights=True,
        mode='max', verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4,
        min_lr=1e-7, verbose=1
    ),
    ModelCheckpoint(
        filepath=CNN_MODEL_PATH,
        monitor='val_auc', save_best_only=True,
        mode='max', verbose=1
    ),
]

history = model.fit(
    train_gen,
    epochs=CNN_EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1,
)

print(f'\n✅ Best model saved to: {CNN_MODEL_PATH}')

## 7. Training Curves

In [ ]:
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics_pairs = [
    ('loss',      'val_loss',      'Loss',      '#ef4444'),
    ('accuracy',  'val_accuracy',  'Accuracy',  '#22c55e'),
    ('auc',       'val_auc',       'AUC',       '#3b82f6'),
    ('recall',    'val_recall',    'Recall',    '#a855f7'),
]

for ax, (tr_key, val_key, title, color) in zip(axes.flat, metrics_pairs):
    ax.plot(epochs_range, hist[tr_key],  label='Train', color=color, linewidth=2)
    ax.plot(epochs_range, hist[val_key], label='Val',   color=color, linewidth=2, linestyle='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('CNN Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'cnn_training_curves.png'), dpi=150)
plt.show()

## 8. Evaluate on Validation Set

In [ ]:
val_gen.reset()
y_prob = model.predict(val_gen, verbose=1)
y_pred = (y_prob > 0.5).astype(int).flatten()
y_true = val_gen.classes
class_labels = list(train_gen.class_indices.keys())

print('\n📊 Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_labels))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'cnn_confusion_matrix.png'), dpi=150)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc_score = roc_auc_score(y_true, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#3b82f6', linewidth=2, label=f'AUC = {auc_score:.4f}')
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Crack Intensity Grading Demo

In [ ]:
import cv2
from src.utils.image_utils import preprocess_for_cnn

crack_class_key = [k for k,v in train_gen.class_indices.items()
                    if any(word in k.lower() for word in ['positive','crack','yes'])]
if not crack_class_key:
    crack_class_key = list(train_gen.class_indices.keys())

crack_dir = class_dirs.get(crack_class_key[0].lower(), list(class_dirs.values())[0])
test_images = [f for f in Path(crack_dir).rglob('*') if f.suffix.lower() in IMG_EXTS][:8]

def grade_intensity(prob):
    if prob >= INTENSITY_HIGH:   return 'HIGH',   '#ef4444'
    elif prob >= INTENSITY_MEDIUM: return 'MEDIUM', '#f59e0b'
    else:                          return 'LOW',    '#22c55e'

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, test_images):
    img_bgr = cv2.imread(str(img_path))
    tensor  = preprocess_for_cnn(img_bgr)
    prob    = float(model.predict(tensor, verbose=0)[0][0])
    label   = 'CRACK' if prob >= 0.5 else 'NO CRACK'
    intensity_label, color = grade_intensity(prob) if prob >= 0.5 else ('—', 'gray')

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f'{label}  [{intensity_label}]\nconf: {prob:.3f}',
                 color=color, fontweight='bold', fontsize=9)
    ax.axis('off')

plt.suptitle('Intensity Grading on Sample Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'saved_models' / 'cnn_intensity_samples.png'), dpi=150)
plt.show()

print(f'\nIntensity thresholds:')
print(f'  HIGH   : confidence >= {INTENSITY_HIGH}')
print(f'  MEDIUM : {INTENSITY_MEDIUM} <= confidence < {INTENSITY_HIGH}')
print(f'  LOW    : confidence < {INTENSITY_MEDIUM}')

## 10. Save & Summary

In [ ]:
# ModelCheckpoint already saved the best model.
# Verify:
assert os.path.exists(CNN_MODEL_PATH), '❌ Model file missing!'
size_mb = os.path.getsize(CNN_MODEL_PATH) / (1024 * 1024)
print(f'✅ CNN model saved: {CNN_MODEL_PATH}')
print(f'   File size: {size_mb:.2f} MB')

# Save metadata
meta = {
    'model': 'CrackCNN',
    'dataset': KAGGLE_DATASET_HANDLE,
    'input_size': list(CNN_IMG_SIZE),
    'intensity_thresholds': {
        'HIGH': INTENSITY_HIGH,
        'MEDIUM': INTENSITY_MEDIUM,
        'LOW': f'< {INTENSITY_MEDIUM}'
    },
    'val_accuracy': float(max(hist['val_accuracy'])),
    'val_auc': float(max(hist['val_auc'])),
}
with open(str(PROJECT_ROOT / 'saved_models' / 'cnn_metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('\n📋 Metadata saved.')
print(json.dumps(meta, indent=2))